In [1]:
#Config 1: Minimal - Single hidden layer
config_1 = {
    'layers': [4],
    'activation': ['relu'],
    'dropout': [0.1]
}

# Config 2: Small but effective
config_2 = {
    'layers': [8, 4], 
    'activations': ['relu', 'relu'],
    'dropout': [0.2, 0.1]
}

# Config 3: Slightly wider
config_3 = {
    'layers': [16, 8], 
    'activations': ['tanh', 'relu'],
    'dropout': [0.3, 0.2]
}

In [2]:
# Config 4: Three layers, moderate size
config_4 = {
    'layers': [16, 8, 4], 
    'activations': ['relu', 'relu', 'relu'],
    'dropout': [0.2, 0.2, 0.1]
}

# Config 5: Uniform layer size
config_5 = {
    'layers': [12, 12, 12], 
    'activations': ['tanh', 'tanh', 'relu'],
    'dropout': [0.2, 0.2, 0.2]
}

# Config 6: Mixed activations
config_6 = {
    'layers': [32, 16, 8], 
    'activations': ['sigmoid', 'relu', 'tanh'],
    'dropout': [0.3, 0.2, 0.1]
}


In [3]:
# Config 7: Deep narrow network
config_7 = {
    'layers': [8, 8, 8, 8, 4], 
    'activations': ['relu', 'relu', 'relu', 'relu', 'relu'],
    'dropout': [0.1, 0.1, 0.1, 0.1, 0.1]
}

# Config 8: Wide then narrow
config_8 = {
    'layers': [64, 32, 16, 4], 
    'activations': ['relu', 'relu', 'tanh', 'relu'],
    'dropout': [0.4, 0.3, 0.2, 0.1]
}


## Implementation Framework

In [34]:
import tensorflow as tf
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, accuracy_score


def create_nn_model(config, input_dim=2):
    """Create NN model based on configuration"""
    model = tf.keras.Sequential()
    
    # Input layer
    model.add(tf.keras.layers.Dense(
        config['layers'][0], 
        activation=config['activations'][0],
        input_shape=(input_dim,)
    ))
    
    # Add dropout if specified
    if 'dropout' in config and len(config['dropout']) > 0:
        model.add(tf.keras.layers.Dropout(config['dropout'][0]))
    
    # Hidden layers
    for i in range(1, len(config['layers'])):
        model.add(tf.keras.layers.Dense(
            config['layers'][i], 
            activation=config['activations'][i]
        ))
        
        if 'dropout' in config and i < len(config['dropout']):
            model.add(tf.keras.layers.Dropout(config['dropout'][i]))
    
    # Output layer (always sigmoid for binary classification)
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    
    return model

def evaluate_nn_config(config, X_train, y_train, X_test, y_test, config_name, model_container):
    """Evaluate a single NN configuration"""
    model = create_nn_model(config)
    model_container[config_name] = model
    # Compile model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    # Train with early stopping
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Evaluate
    y_pred_proba = model.predict(X_test).flatten()
    
    # Calculate metrics
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = auc(recall, precision)
    
    # Get final metrics from training
    final_val_loss = min(history.history['val_loss'])
    epochs_trained = len(history.history['loss'])

    Y_actual = y_test
    Y_prob = y_pred_proba

     # --- Compute best F1 score by sweeping thresholds ---
    thresholds = np.linspace(0, 1, 101)  # thresholds from 0.0 to 1.0
    f1_scores = [f1_score(Y_actual, (Y_prob >= t).astype(int)) for t in thresholds]
    best_f1 = max(f1_scores)
    best_threshold = thresholds[np.argmax(f1_scores)]
    
    # Binary predictions using the best threshold
    Y_pred_best = (Y_prob >= best_threshold).astype(int)
    # --- ROC curve ---
    fpr, tpr, roc_thresholds = roc_curve(Y_actual, Y_prob)
    

    # --- Precision-Recall curve ---
    precision, recall, pr_thresholds = precision_recall_curve(Y_actual, Y_prob)
    avg_precision = average_precision_score(Y_actual, Y_prob)
    acc = accuracy_score(Y_actual, Y_pred_best)

    results = {
        'config_name': config_name,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'val_loss': final_val_loss,
        'epochs': epochs_trained,
        'model': model,
        'history': history,
        "F1": best_f1,
        "Best_Threshold": best_threshold,
        "AUC": roc_auc_score(Y_actual, Y_prob),
        "Average_Precision": avg_precision,
        "Accuracy": acc

    }
    
    return results


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import lightgbm as lgb
import os


df = pd.read_csv("Meta Model Dataset/Training_For_Meta_Model.csv")
# Define target variable (presence of cardiovascular disease)
y = df["Cardiovascular Disease"]

# Select relevant features
features = ["Age", "Systolic Blood Pressure", "Diastolic Blood Pressure", "Cholesterol Level", "Glucose Level", "Smoking Status", "Alcohol Intake", "Physical Activity"]
X = df[features]

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
lifestyle_columns_to_drop = ['Cholesterol Level', 'Diastolic Blood Pressure', 'Systolic Blood Pressure', 'Glucose Level', 'id']
health_columns_to_drop = ['Smoking Status', 'Physical Activity', 'Alcohol Intake','id']

X = df.drop(['Cardiovascular Disease'], axis=1)
Y = df['Cardiovascular Disease']


X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2, 
    random_state=42,
    stratify=Y 
)




health_dataset = X_train.drop(columns=health_columns_to_drop)
lifestyle_dataset = X_train.drop(columns=lifestyle_columns_to_drop)





# Load the trained models
#model_A = lgb.Booster(model_file="saved_models_tausif/lightgbm_model_A.txt")
model_B = lgb.Booster(model_file="Models B/lightgbm_model.txt")
model_A = lgb.Booster(model_file="Saved Models A/lightgbm_model.txt")

predictions_A = model_A.predict(lifestyle_dataset)
predictions_B = model_B.predict(health_dataset)


print(predictions_A)
print(predictions_B)

[0.51450838 0.66875897 0.62748518 ... 0.35488267 0.33029411 0.32365621]
[0.20400517 0.92270298 0.807062   ... 0.22936528 0.58392303 0.22233091]


In [40]:
import numpy as np
# Prepare meta-learner data (probabilities from Model A and B)
meta_X = np.column_stack([predictions_A, predictions_B])

meta_y = Y_train.values

# Split for meta-learner training
X_train_meta, X_test_meta, y_train_meta, y_test_meta = train_test_split(
    meta_X, meta_y, test_size=0.2, random_state=42, stratify=meta_y
)

# Test all configurations
configurations = {
    'Simple_4': {'layers': [4], 'activations': ['relu'], 'dropout': [0.1]},
    'Small_8-4': {'layers': [8, 4], 'activations': ['relu', 'relu'], 'dropout': [0.2, 0.1]},
    'Medium_16-8': {'layers': [16, 8], 'activations': ['tanh', 'relu'], 'dropout': [0.3, 0.2]},
    'Triple_16-8-4': {'layers': [16, 8, 4], 'activations': ['relu', 'relu', 'relu'], 'dropout': [0.2, 0.2, 0.1]},
    'Uniform_12x3': {'layers': [12, 12, 12], 'activations': ['tanh', 'tanh', 'relu'], 'dropout': [0.2, 0.2, 0.2]},
    'Mixed_32-16-8': {'layers': [32, 16, 8], 'activations': ['sigmoid', 'relu', 'tanh'], 'dropout': [0.3, 0.2, 0.1]},
    'Deep_8x5': {'layers': [8, 8, 8, 8, 4], 'activations': ['relu']*5, 'dropout': [0.1]*5},
    'Wide_64-32-16-4': {'layers': [64, 32, 16, 4], 'activations': ['relu', 'relu', 'tanh', 'relu'], 'dropout': [0.4, 0.3, 0.2, 0.1]}
}

results = {}
model_container = {}
for config_name, config in configurations.items():
    print(f"Testing {config_name}...")
    result = evaluate_nn_config(
        config, X_train_meta, y_train_meta, X_test_meta, y_test_meta, config_name, model_container
    )
    results[config_name] = result


Testing Simple_4...


d:\miniConda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Testing Small_8-4...


d:\miniConda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Testing Medium_16-8...


...

69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Testing Deep_8x5...


d:\miniConda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Testing Wide_64-32-16-4...


d:\miniConda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [41]:
import pandas as pd

# Create results comparison
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Configuration': name,
        'ROC-AUC': result['roc_auc'],
        'PR-AUC': result['pr_auc'],
        'Val_Loss': result['val_loss'],
        'Epochs': result['epochs'],
        'F1': result['F1'],
        'Best_Threshold': result['Best_Threshold'],
        'AUC': result['AUC'],
        'Average_Precision': result['Average_Precision'],
        'Accuracy': result['Accuracy']
    })

results_df = pd.DataFrame(comparison_data)
results_df = results_df.sort_values('PR-AUC', ascending=False)
print(results_df)


     Configuration   ROC-AUC    PR-AUC  Val_Loss  Epochs        F1  \
0         Simple_4  0.792096  0.802046  0.553872      96  0.733167   
4     Uniform_12x3  0.792996  0.781631  0.548689      41  0.732811   
3    Triple_16-8-4  0.792507  0.781407  0.551865      40  0.732236   
2      Medium_16-8  0.792669  0.781361  0.549561      74  0.732517   
1        Small_8-4  0.791647  0.780955  0.554610      59  0.728164   
7  Wide_64-32-16-4  0.791705  0.780884  0.548979      54  0.730427   
5    Mixed_32-16-8  0.793037  0.780767  0.551452      44  0.734593   
6         Deep_8x5  0.789069  0.780435  0.568062      20  0.731558   

   Best_Threshold       AUC  Average_Precision  Accuracy  
0            0.36  0.792096           0.777790  0.706178  
4            0.44  0.792996           0.782000  0.718993  
3            0.39  0.792507           0.781867  0.717162  
2            0.43  0.792669           0.781650  0.719908  
1            0.43  0.791647           0.781410  0.720824  
7            0.

In [33]:
best_config_name = "Triple_16-8-4"
best_config = configurations[best_config_name]
best_model = create_nn_model(best_config)

results = {}

print(f"Testing {best_config_name}...")
result = evaluate_nn_config(
    best_config, X_train_meta, y_train_meta, X_test_meta, y_test_meta, best_config_name
)
results[best_config_name] = result  # <-- Use string key here
# ...existing code...
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Configuration': name,
        'ROC-AUC': result['roc_auc'],
        'PR-AUC': result['pr_auc'],
        'Val_Loss': result['val_loss'],
        'Epochs': result['epochs'],
        'F1': result['F1'],
        'Best_Threshold': result['Best_Threshold'],
        'AUC': result['AUC'],
        'Average_Precision': result['Average_Precision'],
        'Accuracy': result['Accuracy']
    })

results_df = pd.DataFrame(comparison_data)
results_df = results_df.sort_values('PR-AUC', ascending=False)
print(results_df)



d:\miniConda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Testing Triple_16-8-4...
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 970us/step
   Configuration   ROC-AUC    PR-AUC  Val_Loss  Epochs        F1  \
0  Triple_16-8-4  0.793116  0.781315  0.549314      61  0.731814   

   Best_Threshold       AUC  Average_Precision  Accuracy  
0            0.42  0.793116           0.781771  0.719908  


In [43]:
model_container['Simple_4'].compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)
#evaluate
model_container['Simple_4'].evaluate(X_test_meta, y_test_meta)
#save the model
model_container['Simple_4'].save('simple_4_model.h5')

69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7281 - loss: 0.5585 - precision: 0.7431 - recall: 0.6910  
